In [2]:
from pathlib import Path
import pandas as pd

workspace_root = (Path.cwd() / '..').resolve()

# Target JSON (for this notebook context)
json_path = workspace_root / 'JSON Whole Model' / 'Ifc2x3_Duplex_Architecture.json'
assert json_path.exists(), f'JSON not found: {json_path}'

# Read Excel from COBie folder (columns G and I)
excel_path = workspace_root / 'COBie' / 'Uniclass2015_EF_v1_16.xlsx'
assert excel_path.exists(), f'Excel not found: {excel_path}'

# Read only columns G and I
df = pd.read_excel(excel_path, usecols='G,I')
df.columns = ['G', 'I']

# Find rows where Column G == 'Walls'
walls_rows = df[df['G'].astype(str).str.strip().str.casefold() == 'walls']

if walls_rows.empty:
    print("No match found for 'Walls' in column G.")
else:
    first_value = walls_rows['I'].iloc[0]
    print('First corresponding value in column I for Walls:')
    print(first_value)

First corresponding value in column I for Walls:
EF_25_10 : Walls


In [3]:
import json
import pandas as pd

# Load JSON Whole Model data
with json_path.open('r', encoding='utf-8') as f:
    model_data = json.load(f)

def find_wall_property_matches(properties):
    matches = []
    if not isinstance(properties, list):
        return matches

    for prop in properties:
        if not isinstance(prop, dict):
            continue
        haystack_parts = [
            str(prop.get('category', '')),
            str(prop.get('displayName', '')),
            str(prop.get('value', ''))
        ]
        haystack = ' | '.join(haystack_parts).lower()
        if 'wall' in haystack:
            matches.append(prop)
    return matches

result_rows = []
for item in model_data:
    if not isinstance(item, dict):
        continue

    properties = item.get('Properties')
    wall_matches = find_wall_property_matches(properties)
    if not wall_matches:
        continue

    result_rows.append({
        'Name': item.get('Name'),
        'Dbid': item.get('DbId'),
        'WallPropertyMatch': len(wall_matches)
    })

result_table = pd.DataFrame(result_rows)

if result_table.empty:
    print("No objects found with 'Wall' in Properties.")
else:
    result_table['Object Count'] = result_table.groupby('Name')['Name'].transform('count')
    result_table = (
        result_table[['Name', 'Dbid', 'Object Count', 'WallPropertyMatch']]
        .sort_values(['Name', 'Dbid'], kind='stable')
        .reset_index(drop=True)
    )

    print(f"Total rows in result table: {len(result_table)}")
    print("Showing up to 200 rows:")

    display_table = result_table.head(200).reset_index(drop=True)
    with pd.option_context('display.max_rows', 200, 'display.min_rows', 200):
        display(display_table)

Total rows in result table: 231
Showing up to 200 rows:


,Name,Dbid,Object Count,WallPropertyMatch
0,A102,9,1,2
1,A103,10,1,2
2,A104,11,1,2
3,A202,677,1,2
4,A203,676,1,2
5,A204,675,1,2
6,B102,13,1,2
7,B103,14,1,2
8,B202,681,1,2
9,B203,680,1,2


In [4]:
from pathlib import Path
from shutil import copy2
import json

# Resolve COBie value from Cell 1 output/variables
if 'first_value' in globals() and str(first_value).strip():
    cobie_value = str(first_value).strip()
else:
    walls_rows_local = df[df['G'].astype(str).str.strip().str.casefold() == 'walls']
    assert not walls_rows_local.empty, "Could not find 'Walls' in Excel column G."
    cobie_value = str(walls_rows_local['I'].iloc[0]).strip()

# Ensure destination backup folder exists
json_edit_dir = workspace_root / 'JSON_Edit'
json_edit_dir.mkdir(parents=True, exist_ok=True)

# Step 4: backup original JSON to JSON_Edit with same name
backup_json_path = json_edit_dir / json_path.name
copy2(json_path, backup_json_path)

# Step 5: edit the target JSON file (original path)
with json_path.open('r', encoding='utf-8') as f:
    data_to_update = json.load(f)

def get_prop_value(properties, category, display_name):
    if not isinstance(properties, list):
        return None
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        if str(prop.get('category', '')).strip().upper() == category and str(prop.get('displayName', '')).strip().upper() == display_name:
            return str(prop.get('value', '')).strip()
    return None

def set_or_add_cobie(properties, new_value):
    if not isinstance(properties, list):
        return False, False, None, None

    before_value = None
    after_value = None
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        if str(prop.get('category', '')).strip() == 'IFC' and str(prop.get('displayName', '')).strip() == 'COBie':
            before_value = str(prop.get('value', '')).strip()
            prop['value'] = new_value
            after_value = str(prop.get('value', '')).strip()
            return True, True, before_value, after_value

    properties.append({'category': 'IFC', 'displayName': 'COBie', 'value': new_value})
    after_value = new_value
    return True, False, before_value, after_value

ifcwall_exact_count = 0
ifcwall_family_count = 0
updated_count = 0
added_count = 0
changed_count = 0
before_values = []
after_values = []

for item in data_to_update:
    if not isinstance(item, dict):
        continue

    properties = item.get('Properties')
    if not isinstance(properties, list):
        continue

    object_type_value = get_prop_value(properties, 'ITEM', 'TYPE')
    object_type_upper = (object_type_value or '').upper()

    if object_type_upper == 'IFCWALL':
        ifcwall_exact_count += 1

    if not object_type_upper.startswith('IFCWALL'):
        continue

    ifcwall_family_count += 1

    found_or_added, was_update, before_value, after_value = set_or_add_cobie(properties, cobie_value)
    if found_or_added:
        updated_count += 1
        before_values.append(before_value if before_value is not None else '(missing)')
        after_values.append(after_value if after_value is not None else '(missing)')
        if was_update:
            if before_value != after_value:
                changed_count += 1
        else:
            added_count += 1

with json_path.open('w', encoding='utf-8') as f:
    json.dump(data_to_update, f, ensure_ascii=False, indent=2)

print(f'Backup created at: {backup_json_path}')
print(f'IFCWALL exact elements found (Item/Type=IFCWALL): {ifcwall_exact_count}')
print(f'IFCWALL* elements found (Item/Type startswith IFCWALL): {ifcwall_family_count}')
print(f'Elements processed for COBie: {updated_count}')
print(f'Existing COBie values changed: {changed_count}')
print(f'COBie objects added: {added_count}')
print(f'COBie value applied from Excel (Walls): {cobie_value}')

before_unique = sorted(set(before_values)) if before_values else []
after_unique = sorted(set(after_values)) if after_values else []
print(f'Before COBie values (unique): {before_unique}')
print(f'After COBie values (unique): {after_unique}')

Backup created at: C:\Git\APS-IFC\JSON_Edit\Ifc2x3_Duplex_Architecture.json
IFCWALL exact elements found (Item/Type=IFCWALL): 1
IFCWALL* elements found (Item/Type startswith IFCWALL): 57
Elements processed for COBie: 57
Existing COBie values changed: 0
COBie objects added: 0
COBie value applied from Excel (Walls): EF_25_10 : Walls
Before COBie values (unique): ['EF_25_10 : Walls']
After COBie values (unique): ['EF_25_10 : Walls']


In [5]:
from pathlib import Path
from shutil import copy2
import json
import pandas as pd
from IPython.display import display

workspace_root = (Path.cwd() / '..').resolve()
source_json_path = workspace_root / 'JSON Whole Model' / 'Ifc2x3_Duplex_Architecture.json'
assert source_json_path.exists(), f'JSON not found: {source_json_path}'

json_edit_dir = workspace_root / 'JSON_Edit'
json_edit_dir.mkdir(parents=True, exist_ok=True)
working_json_path = json_edit_dir / source_json_path.name

if not working_json_path.exists():
    copy2(source_json_path, working_json_path)
    print(f'Copied source JSON to: {working_json_path}')

excel_path = workspace_root / 'COBie' / 'Uniclass2015_EF_v1_16.xlsx'
assert excel_path.exists(), f'Excel not found: {excel_path}'

# --- Excel search: read sheet 'EF' (header row 2) to get Code, Title, COBie columns ---
ef_df = pd.read_excel(excel_path, sheet_name='EF', header=2)

# --- Excel return (Doors): rows where Title contains 'door' (case-insensitive) ---
door_title_df = ef_df[ef_df['Title'].astype(str).str.contains('door', case=False, na=False)].copy()
door_title_df = door_title_df.sort_values(by=['Title', 'Code'], kind='stable').reset_index(drop=True)
assert not door_title_df.empty, "Could not find 'Doors' row in COBie Excel."
cobie_value = str(door_title_df.iloc[0]['COBie']).strip()

with source_json_path.open('r', encoding='utf-8') as f:
    source_data = json.load(f)

with working_json_path.open('r', encoding='utf-8') as f:
    working_data = json.load(f)


def is_ifcdoor(item):
    if not isinstance(item, dict):
        return False

    properties = item.get('Properties', [])
    if not isinstance(properties, list):
        return False

    for prop in properties:
        if not isinstance(prop, dict):
            continue

        category = str(prop.get('category', '')).strip().lower()
        display_name = str(prop.get('displayName', '')).strip().lower()
        value = str(prop.get('value', '')).strip().upper()

        if category == 'item' and display_name == 'type' and value == 'IFCDOOR':
            return True

    return False


door_rows = []
for item in source_data:
    if is_ifcdoor(item):
        door_rows.append(
            {
                'GUID': str(item.get('ExternalId', '')).strip(),
                'Name': item.get('Name', ''),
                'DbId': item.get('DbId'),
                'Item.Type': 'IFCDOOR',
            }
        )

door_df = pd.DataFrame(door_rows)

if door_df.empty:
    print('No IFCDOOR objects found in the IFC model JSON.')
else:
    door_df = door_df.sort_values(by=['Name', 'GUID'], kind='stable').reset_index(drop=True)
    door_df.insert(0, 'Count', door_df.index + 1)

    working_by_guid = {}
    for item in working_data:
        if not isinstance(item, dict):
            continue
        guid = str(item.get('ExternalId', '')).strip()
        if guid:
            working_by_guid[guid] = item

    updated_count = 0
    added_count = 0
    missing_guid_count = 0

    for row in door_df.itertuples(index=False):
        target_item = working_by_guid.get(str(row.GUID).strip())
        if not isinstance(target_item, dict):
            missing_guid_count += 1
            continue

        properties = target_item.get('Properties')
        if not isinstance(properties, list):
            properties = []
            target_item['Properties'] = properties

        existing_cobie_prop = None
        for prop in properties:
            if not isinstance(prop, dict):
                continue
            if (
                str(prop.get('category', '')).strip() == 'IFC'
                and str(prop.get('displayName', '')).strip() == 'COBie'
            ):
                existing_cobie_prop = prop
                break

        if existing_cobie_prop is None:
            properties.append(
                {
                    'category': 'IFC',
                    'displayName': 'COBie',
                    'value': cobie_value,
                }
            )
            added_count += 1
            updated_count += 1
        else:
            old_value = str(existing_cobie_prop.get('value', '')).strip()
            if old_value != cobie_value:
                existing_cobie_prop['value'] = cobie_value
                updated_count += 1

    with working_json_path.open('w', encoding='utf-8') as f:
        json.dump(working_data, f, ensure_ascii=False, indent=2)

    door_df['COBie (from Excel)'] = cobie_value

    door_count_summary_df = pd.DataFrame(
        [
            {
                'Model': 'IFC model JSON (before change)',
                'Filter': 'category=Item, displayName=Type, value=IFCDOOR',
                'Door Count': len(door_df),
                'COBie Title Rows (Doors)': len(door_title_df),
                'COBie Value Applied': cobie_value,
                'JSON_Edit Updated Objects': updated_count,
                'JSON_Edit Added IFC/COBie': added_count,
                'Missing GUID Matches': missing_guid_count,
                'JSON_Edit Path': str(working_json_path),
                'Total Records': len(source_data),
            }
        ]
    )

    print(f'Updated JSON written to: {working_json_path}')
    print(f'Total IFCDOOR transformed from JSON: {len(door_df)}')
    print(f'Total records in source JSON: {len(source_data)}')
    display(door_count_summary_df)
    display(door_title_df[['Code', 'Title', 'COBie']])
    display(door_df[['Count', 'GUID', 'Name', 'DbId', 'Item.Type', 'COBie (from Excel)']])

# ============================================================
# --- WINDOWS SECTION ---
# ============================================================

# --- Excel search: rows where Title contains 'windows' (case-insensitive) ---
# Targets COBie code EF_25_30_97 : Windows from the EF sheet
window_title_df = ef_df[ef_df['Title'].astype(str).str.contains('windows', case=False, na=False)].copy()
window_title_df = window_title_df.sort_values(by=['Title', 'Code'], kind='stable').reset_index(drop=True)
assert not window_title_df.empty, "Could not find 'Windows' row in COBie Excel."

# --- Excel return (Windows): COBie value from column 'COBie' for Windows row ---
# Expected: EF_25_30_97 : Windows
window_cobie_value = str(window_title_df.iloc[0]['COBie']).strip()

# Re-read working JSON to apply Windows on top of the Door changes already written
with working_json_path.open('r', encoding='utf-8') as f:
    working_data_win = json.load(f)

# Build GUID lookup for working data
working_by_guid_win = {}
for item in working_data_win:
    if not isinstance(item, dict):
        continue
    guid = str(item.get('ExternalId', '')).strip()
    if guid:
        working_by_guid_win[guid] = item

# --- JSON search: find all items where category='item', displayName='type', value='IFCWINDOW' ---
# (Object Type = IFCWINDOW in the Properties array of the source JSON)
window_rows = []
for item in source_data:
    if not isinstance(item, dict):
        continue
    for prop in item.get('Properties', []):
        if not isinstance(prop, dict):
            continue
        if (
            str(prop.get('category', '')).strip().lower() == 'item'
            and str(prop.get('displayName', '')).strip().lower() == 'type'
            and str(prop.get('value', '')).strip().upper() == 'IFCWINDOW'
        ):
            window_rows.append({
                'GUID': str(item.get('ExternalId', '')).strip(),
                'Name': item.get('Name', ''),
                'DbId': item.get('DbId'),
                'Item.Type': 'IFCWINDOW',
            })
            break

window_df = pd.DataFrame(window_rows)
window_before_count = len(window_df)
print(f'\n--- Windows ---')
print(f'IFCWINDOW objects found in source JSON (before): {window_before_count}')

if window_df.empty:
    print('No IFCWINDOW objects found in the IFC model JSON.')
else:
    window_df = window_df.sort_values(by=['Name', 'GUID'], kind='stable').reset_index(drop=True)
    window_df.insert(0, 'Count', window_df.index + 1)

    win_updated_count = 0
    win_added_count = 0
    win_missing_guid_count = 0
    win_before_values = []
    win_after_values = []

    for row in window_df.itertuples(index=False):
        target_item = working_by_guid_win.get(str(row.GUID).strip())
        if not isinstance(target_item, dict):
            win_missing_guid_count += 1
            continue

        properties = target_item.get('Properties')
        if not isinstance(properties, list):
            properties = []
            target_item['Properties'] = properties

        existing_cobie_prop = None
        for prop in properties:
            if not isinstance(prop, dict):
                continue
            if (
                str(prop.get('category', '')).strip() == 'IFC'
                and str(prop.get('displayName', '')).strip() == 'COBie'
            ):
                existing_cobie_prop = prop
                break

        if existing_cobie_prop is None:
            # Before: no COBie property exists on this IFCWINDOW item
            win_before_values.append('(none)')
            properties.append({'category': 'IFC', 'displayName': 'COBie', 'value': window_cobie_value})
            win_after_values.append(window_cobie_value)
            win_added_count += 1
            win_updated_count += 1
        else:
            # Before: existing COBie value recorded before overwrite
            old_value = str(existing_cobie_prop.get('value', '')).strip()
            win_before_values.append(old_value if old_value else '(empty)')
            if old_value != window_cobie_value:
                existing_cobie_prop['value'] = window_cobie_value
                win_updated_count += 1
            win_after_values.append(window_cobie_value)

    with working_json_path.open('w', encoding='utf-8') as f:
        json.dump(working_data_win, f, ensure_ascii=False, indent=2)

    window_df['COBie Before'] = win_before_values
    window_df['COBie After'] = win_after_values

    win_before_unique = sorted(set(win_before_values))
    win_after_unique = sorted(set(win_after_values))

    print(f'IFCWINDOW objects processed (after): {win_updated_count}')
    print(f'COBie entries added (new): {win_added_count}')
    print(f'Missing GUID matches: {win_missing_guid_count}')
    print(f'COBie value applied from Excel (Windows): {window_cobie_value}')
    print(f'Before COBie values (unique): {win_before_unique}')
    print(f'After COBie values (unique):  {win_after_unique}')
    print(f'Updated JSON written to: {working_json_path}')

    win_summary_df = pd.DataFrame([{
        'Model': 'IFC model JSON (before change)',
        'Filter': 'category=Item, displayName=Type, value=IFCWINDOW',
        'Window Count (Before)': window_before_count,
        'COBie Title Rows (Windows)': len(window_title_df),
        'COBie Value Applied': window_cobie_value,
        'JSON_Edit Updated Objects': win_updated_count,
        'JSON_Edit Added IFC/COBie': win_added_count,
        'Missing GUID Matches': win_missing_guid_count,
    }])
    display(win_summary_df)
    display(window_title_df[['Code', 'Title', 'COBie']])
    display(window_df[['Count', 'GUID', 'Name', 'DbId', 'Item.Type', 'COBie Before', 'COBie After']])


Updated JSON written to: C:\Git\APS-IFC\JSON_Edit\Ifc2x3_Duplex_Architecture.json
Total IFCDOOR transformed from JSON: 14
Total records in source JSON: 1432


,Model,Filter,Door Count,COBie Title Rows (Doors),COBie Value Applied,JSON_Edit Updated Objects,JSON_Edit Added IFC/COBie,Missing GUID Matches,JSON_Edit Path,Total Records
0,IFC model JSON (before change),"category=Item, displayName=Type, value=IFCDOOR",14,1,EF_25_30_25 : Doors,14,14,0,C:\Git\APS-IFC\JSON_Edit\Ifc2x3_Duplex_Archite...,1432


,Code,Title,COBie
0,EF_25_30_25,Doors,EF_25_30_25 : Doors


,Count,GUID,Name,DbId,Item.Type,COBie (from Excel)
0,1,0/0/0/0/32,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150173,41,IFCDOOR,EF_25_30_25 : Doors
1,2,0/0/0/0/33,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150257,42,IFCDOOR,EF_25_30_25 : Doors
2,3,0/0/0/1/68,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:203720,742,IFCDOOR,EF_25_30_25 : Doors
3,4,0/0/0/1/70,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:204034,744,IFCDOOR,EF_25_30_25 : Doors
4,5,0/0/0/1/33,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:150378,707,IFCDOOR,EF_25_30_25 : Doors
5,6,0/0/0/1/34,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:150478,708,IFCDOOR,EF_25_30_25 : Doors
6,7,0/0/0/1/35,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:159734,709,IFCDOOR,EF_25_30_25 : Doors
7,8,0/0/0/1/36,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:159834,710,IFCDOOR,EF_25_30_25 : Doors
8,9,0/0/0/1/37,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:160065,711,IFCDOOR,EF_25_30_25 : Doors
9,10,0/0/0/1/38,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:160208,712,IFCDOOR,EF_25_30_25 : Doors



--- Windows ---
IFCWINDOW objects found in source JSON (before): 24
IFCWINDOW objects processed (after): 24
COBie entries added (new): 24
Missing GUID matches: 0
COBie value applied from Excel (Windows): EF_25_30_97 : Windows
Before COBie values (unique): ['(none)']
After COBie values (unique):  ['EF_25_30_97 : Windows']
Updated JSON written to: C:\Git\APS-IFC\JSON_Edit\Ifc2x3_Duplex_Architecture.json


,Model,Filter,Window Count (Before),COBie Title Rows (Windows),COBie Value Applied,JSON_Edit Updated Objects,JSON_Edit Added IFC/COBie,Missing GUID Matches
0,IFC model JSON (before change),"category=Item, displayName=Type, value=IFCWINDOW",24,1,EF_25_30_97 : Windows,24,24,0


,Code,Title,COBie
0,EF_25_30_97,Windows,EF_25_30_97 : Windows


,Count,GUID,Name,DbId,Item.Type,COBie Before,COBie After
0,1,0/0/0/1/27,M_Casement:819mm x 759mm:819mm x 759mm:148607,701,IFCWINDOW,(none),EF_25_30_97 : Windows
1,2,0/0/0/1/31,M_Casement:819mm x 759mm:819mm x 759mm:149736,705,IFCWINDOW,(none),EF_25_30_97 : Windows
2,3,0/0/0/1/50,M_Casement:819mm x 759mm:819mm x 759mm:180994,724,IFCWINDOW,(none),EF_25_30_97 : Windows
3,4,0/0/0/1/53,M_Casement:819mm x 759mm:819mm x 759mm:181548,727,IFCWINDOW,(none),EF_25_30_97 : Windows
4,5,0/0/0/1/25,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:147686,699,IFCWINDOW,(none),EF_25_30_97 : Windows
5,6,0/0/0/1/29,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:149278,703,IFCWINDOW,(none),EF_25_30_97 : Windows
6,7,0/0/0/1/47,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:180318,721,IFCWINDOW,(none),EF_25_30_97 : Windows
7,8,0/0/0/1/51,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:181096,725,IFCWINDOW,(none),EF_25_30_97 : Windows
8,9,0/0/0/0/26,M_Fixed:4835mm x 2420mm:4835mm x 2420mm:145788,35,IFCWINDOW,(none),EF_25_30_97 : Windows
9,10,0/0/0/0/27,M_Fixed:4835mm x 2420mm:4835mm x 2420mm:146016,36,IFCWINDOW,(none),EF_25_30_97 : Windows
